In [4]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    provider="auto",
    max_new_tokens=256
)

model = ChatHuggingFace(llm=llm)

In [7]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [ ]:
def create_outline(state:BlogState) -> BlogState:
    # fetch title

     title = state['title']

     #call llm to generate outline
     
     prompt = f'Generate a detailed outline for a blog on topic {title}'
     outline = model.invoke(prompt).content
     
     # update state
     state['outline'] = outline

     return state

In [10]:
def create_blog(state:BlogState)-> BlogState:

    # fetch outline and title
    title = state['title']
    outline = state['outline']

    # call llm to generate blog

    prompt = f'Write a detailed blog on the topic - {title} using the following outline \n {outline} '
    content = model.invoke(prompt).content
    # update state
    state['content'] = content

    return state

In [15]:
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow = graph.compile()


In [16]:
initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)
print(final_state)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ae44c73-7b32fa3616e9556770160737;0929d422-9bc8-4b7f-92eb-4bab1ca1e2c7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.